# Synthetic INDEL Demo

Notebook didatico para auditar o pre-processamento de mutacoes/INDELs em um exemplo minimo: 10 bp, 2 individuos e 5 mutacoes.

Objetivo:

- entender como o VCF left-anchored e interpretado
- ver como o eixo expandido e construido
- conferir manualmente se `values`, `ins_mask`, `del_mask` e `valid_mask` batem com a expectativa humana


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != 'notebooks':
    NOTEBOOK_DIR = NOTEBOOK_DIR / 'genotype_based_predictor' / 'notebooks'
REPO_ROOT = NOTEBOOK_DIR.parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from genotype_based_predictor.synthetic_indel_demo import build_demo_global_tensor


In [2]:
demo = build_demo_global_tensor()
display({
    'reference': demo['reference'],
    'track': demo['track'].tolist(),
    'sample_ids': demo['sample_ids'],
    'haplotypes': demo['haplotypes'],
    'tensor_shape': tuple(demo['tensor'].shape),
    'expanded_length': demo['alignment']['expanded_length'],
})


{'reference': 'ACGTACGTAA',
 'track': [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0],
 'sample_ids': ['IND1', 'IND2'],
 'haplotypes': ['H1', 'H2'],
 'tensor_shape': (2, 2, 4, 12),
 'expanded_length': 12}

## VCF simplificado

Este exemplo contem 4 mutacoes, incluindo:

- SNP
- delecao simples
- insercao simples
- outra SNP


In [3]:
display(demo['variants_df'])


,pos_1based,ref,alt,vcf_type,IND1,IND2
0,2,C,T,same_length,0|1,1|1
1,4,TA,T,length_change,1|1,0|0
2,6,C,CGG,length_change,0|1,1|0
3,8,T,G,same_length,1|0,0|1


## Sequencias haplotipicas resultantes

As sequencias abaixo permitem verificar se os alelos foram aplicados como esperado em cada haplotipo.


In [4]:
hap_rows = []
for (sample_id, haplotype), sequence in demo['hap_sequences'].items():
    hap_rows.append({'sample_id': sample_id, 'haplotype': haplotype, 'sequence': sequence, 'length': len(sequence)})
display(pd.DataFrame(hap_rows))


,sample_id,haplotype,sequence,length
0,IND1,H1,ACGTCGGAA,9
1,IND1,H2,ATGTCGGGTAA,11
2,IND2,H1,ATGTACGGGTAA,12
3,IND2,H2,ATGTACGGAA,10


## Mapeamento dinamico por haplotipo

O alinhamento dinamico gera, para cada amostra/haplotipo:

- `copy_from_indices`
- `expanded_indices`
- `insertion_indices`
- `deletion_indices`

Esses artefatos sao a base do tensor.


In [5]:
for sample_id in demo['sample_ids']:
    for haplotype in demo['haplotypes']:
        entry = demo['alignment']['samples'][sample_id][haplotype]
        print(f'{sample_id} {haplotype}')
        display({
            'copy_from_indices': entry['copy_from_indices'],
            'expanded_indices': entry['expanded_indices'],
            'insertion_indices': entry['insertion_indices'],
            'deletion_indices': entry['deletion_indices'],
        })


IND1 H1


{'copy_from_indices': [0, 1, 2, 3, 4, 5, 6, 7, 8],
 'expanded_indices': [0, 1, 2, 3, 5, 8, 9, 10, 11],
 'insertion_indices': [],
 'deletion_indices': [4]}

IND1 H2


{'copy_from_indices': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
 'expanded_indices': [0, 1, 2, 3, 5, 6, 7, 8, 9, 10, 11],
 'insertion_indices': [6, 7],
 'deletion_indices': [4]}

IND2 H1


{'copy_from_indices': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11],
 'expanded_indices': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11],
 'insertion_indices': [6, 7],
 'deletion_indices': []}

IND2 H2


{'copy_from_indices': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
 'expanded_indices': [0, 1, 2, 3, 4, 5, 8, 9, 10, 11],
 'insertion_indices': [],
 'deletion_indices': []}

## Tensor final

Cada amostra gera um tensor `(2, 4, L)`:

- eixo 0: `H1`, `H2`
- eixo 1: `values`, `ins_mask`, `del_mask`, `valid_mask`
- eixo 2: posicoes no eixo expandido

Abaixo mostramos isso como matriz para auditoria humana.


In [6]:
CHANNEL_NAMES = ['values', 'ins_mask', 'del_mask', 'valid_mask']
columns = list(range(demo['alignment']['expanded_length']))
for sample_idx, sample_id in enumerate(demo['sample_ids']):
    rows = []
    for hap_idx, haplotype in enumerate(demo['haplotypes']):
        for channel_idx, channel_name in enumerate(CHANNEL_NAMES):
            values = demo['tensor'][sample_idx, hap_idx, channel_idx]
            if channel_name == 'values':
                row_values = np.round(values.astype(np.float32), 3).tolist()
            else:
                row_values = values.astype(int).tolist()
            payload = {'sample_id': sample_id, 'haplotype': haplotype, 'channel': channel_name}
            payload.update({col: value for col, value in zip(columns, row_values)})
            rows.append(payload)
    print(f'Tensor de {sample_id}')
    display(pd.DataFrame(rows))


Tensor de IND1


,sample_id,haplotype,channel,0,1,2,3,4,5,6,7,8,9,10,11
0,IND1,H1,values,1.0,2.0,3.0,4.0,0.0,6.0,0.0,0.0,7.0,8.0,9.0,10.0
1,IND1,H1,ins_mask,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,IND1,H1,del_mask,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,IND1,H1,valid_mask,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0
4,IND1,H2,values,1.0,2.0,3.0,4.0,0.0,6.0,6.0,6.0,7.0,8.0,9.0,10.0
5,IND1,H2,ins_mask,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
6,IND1,H2,del_mask,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,IND1,H2,valid_mask,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


Tensor de IND2


,sample_id,haplotype,channel,0,1,2,3,4,5,6,7,8,9,10,11
0,IND2,H1,values,1.0,2.0,3.0,4.0,5.0,6.0,6.0,6.0,7.0,8.0,9.0,10.0
1,IND2,H1,ins_mask,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
2,IND2,H1,del_mask,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,IND2,H1,valid_mask,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
4,IND2,H2,values,1.0,2.0,3.0,4.0,5.0,6.0,0.0,0.0,7.0,8.0,9.0,10.0
5,IND2,H2,ins_mask,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,IND2,H2,del_mask,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,IND2,H2,valid_mask,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0


## Casos faceis de conferir

Use estas perguntas para auditar o resultado:

1. Para `TA -> T`, a coluna marcada em `del_mask` corresponde apenas a base deletada, nao a ancora?
2. Para `C -> CGG`, aparecem exatamente 2 colunas em `ins_mask` para os haplotipos que carregam ALT?
3. Para SNPs, `ins_mask` e `del_mask` ficam zerados?
4. Para `<DEL>`, o tratamento ficou coerente com a convencao left-anchored usada pelo pipeline?
5. `valid_mask` marca exatamente as colunas que receberam copia de `track`?
